### Compactación mediante LIQUID CLUSTERING
Liquid Clustering es una forma automática y flexible de organizar los datos en tablas Delta que reordena los archivos internamente según las columnas más consultadas, sin crear particiones físicas rígidas.
> 1. Crear **500,000,000** de registros aleatorios
> 2. Guardar los registros en una tabla llamada "**demo.delta_lake.order_data_liquid_clustering**"
> 3. Consultar la tabla sin compactación "**LIQUID CLUSTERING**"
> 4. Ejecutar "**LIQUID CLUSTERING**" (city, order_date)"
> 5. Consultar la tabla con compactación "**LIQUID CLUSTERING**"

#### 1. Crear **500,000,000** de registros aleatorios

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# Número de registros
num_records = 500000000

# Paso 1 - Crear un Dataframe con 50,000 registros
df = spark.range(0, num_records)

# Paso 2 - Agregar las columnas requeridas
df = (
    df.withColumn("order_id", F.concat(F.lit("ODR_"), F.lpad(F.col("id"), 9, "0")))
      .withColumn("city", F.when(F.rand() < 0.4, "New York")
                           .when(F.rand() < 0.7, "San Diego")
                           .otherwise("Dallas"))
      .withColumn("order_date",
                  F.date_add(F.lit("2026-01-10"), (F.rand() * 5).cast("int")))
      .withColumn("weight", F.round((F.rand() * 10).cast("double"), 4))
      .drop("id")
)

df.show(10)
print("Total records:", df.count())

#### 2. Guardar los registros en una tabla llamada "**demo.delta_lake.order_data_liquid_clustering**"

In [0]:
%python
df.writeTo("demo.delta_lake.order_data_liquid_clustering").createOrReplace()

#### 3. Consultar la tabla sin compactación "**LIQUID CLUSTERING**"

In [0]:
DESCRIBE DETAIL demo.delta_lake.order_data_liquid_clustering;

In [0]:
SELECT * FROM demo.delta_lake.order_data_liquid_clustering
WHERE city = 'New York' AND order_date = '2026-01-10';

#### 4. Ejecutar "**LIQUID CLUSTERING**" (city, order_date)**"

In [0]:
ALTER TABLE demo.delta_lake.order_data_liquid_clustering
CLUSTER BY (city, order_date);

In [0]:
OPTIMIZE demo.delta_lake.order_data_liquid_clustering;

#### 5. Consultar la tabla con compactación "**LIQUID CLUSTERING**"

In [0]:
DESCRIBE DETAIL demo.delta_lake.order_data_liquid_clustering;

In [0]:
SELECT * FROM demo.delta_lake.order_data_liquid_clustering
WHERE city = 'New York' AND order_date = '2026-01-10';